In [3]:

import torch
import torch.nn as nn
from torchvision import models, transforms

import time
import copy
import random
import numpy as np
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# Untuk reproduksibilitas
def set_seed(seed=67):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(67)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================================================

# 1. Inisialisasi Model & Ekstraksi Bobot Pretrained
weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights)

# 2. Konfigurasi B2: Feature Extraction (Bekukan Backbone)
for param in model.parameters():
  param.requires_grad = False

# Pindahin model ke GPU untuk alamatnya diallocate
model = model.to(device)

# Ganti classifier head untuk 10 kelas
num_features = model. fc.in_features
model.fc = nn.Linear(num_features, 10)

### TAMBAHAN: Pindahkan model ke GPU sebelum mendefinisikan optimizer ###
model = model.to(device)

# 3. Konfigurasi B3: Unfreeze Blok Residual Terakhir (layer4)
for param in model. layer4.parameters():
  param.requires_grad = True

# Optimasi dengan Differential Learning Rate
optimizer = torch. optim. AdamW([
    {"params": model. layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
    ], weight_decay=1e-2)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s]


In [4]:
# 4. Pipeline Transformasi Data
# Data latih: menerapkan augmentasi aktif
train_transforms = transforms . Compose ([
  transforms. Resize(256),
  transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
  transforms. RandomHorizontalFlip(p=0.5),
  transforms. RandomRotation(degrees=15),
  transforms.ColorJitter(brightness=0.2, contrast=0.2),
  transforms. ToTensor(),
  transforms. Normalize(mean=weights.transforms().mean,
  std=weights.transforms().std),])

# Data validasi/uji: hanya deterministik tanpa distorsi
eval_transforms = weights.transforms()

# Verifikasi parameter yang dapat dilatih
trainable_params = sum(p.numel() for p in model.parameters() if
p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable Parameters: {trainable_params:,} / {total_params:,}")

Trainable Parameters: 8,398,858 / 11,181,642


In [5]:
# 5. Persiapan Dataset (Oxford-IIIT Pet - 10 Kelas)
print("\nMengunduh dan menyiapkan dataset...")
full_dataset = datasets.OxfordIIITPet(root='./data', split='trainval', download=True)
target_classes = list(range(10))
valid_indices = [i for i, target in enumerate(full_dataset._labels) if target in target_classes]
valid_targets = [full_dataset._labels[i] for i in valid_indices]

# Stratified Split: 70% Train, 15% Val, 15% Test
train_idx, temp_idx, _, temp_targets = train_test_split(valid_indices, valid_targets, test_size=0.3, stratify=valid_targets, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=temp_targets, random_state=42)

class PetSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset, self.indices, self.transform = dataset, indices, transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        return self.transform(img) if self.transform else img, label

# DataLoader Batch Size 32
train_loader = DataLoader(PetSubset(full_dataset, train_idx, train_transforms), batch_size=32, shuffle=True)
val_loader = DataLoader(PetSubset(full_dataset, val_idx, eval_transforms), batch_size=32, shuffle=False)
test_loader = DataLoader(PetSubset(full_dataset, test_idx, eval_transforms), batch_size=32, shuffle=False)


Mengunduh dan menyiapkan dataset...


100%|██████████| 792M/792M [00:54<00:00, 14.4MB/s]
100%|██████████| 19.2M/19.2M [00:02<00:00, 9.38MB/s]


In [6]:
# 6. Training Loop dengan Early Stopping[cite: 4]
criterion = nn.CrossEntropyLoss()
num_epochs = 50
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_wts = copy.deepcopy(model.state_dict())

print("\nMemulai Pelatihan...")
for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct = 0.0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
        train_correct += torch.sum(torch.max(outputs, 1)[1] == labels.data)

    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            val_correct += torch.sum(torch.max(outputs, 1)[1] == labels.data)

    ep_t_loss, ep_t_acc = train_loss / len(train_idx), train_correct.double() / len(train_idx)
    ep_v_loss, ep_v_acc = val_loss / len(val_idx), val_correct.double() / len(val_idx)

    print(f"Epoch {epoch+1:02d} | Train Loss: {ep_t_loss:.4f}, Acc: {ep_t_acc:.4f} | Val Loss: {ep_v_loss:.4f}, Acc: {ep_v_acc:.4f}")

    # Early Stopping[cite: 4]
    if ep_v_loss < best_val_loss:
        best_val_loss, best_model_wts, epochs_no_improve = ep_v_loss, copy.deepcopy(model.state_dict()), 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping terpicu di epoch {epoch+1}!")
            break


Memulai Pelatihan...
Epoch 01 | Train Loss: 1.2678, Acc: 0.6356 | Val Loss: 0.3915, Acc: 0.8725
Epoch 02 | Train Loss: 0.3090, Acc: 0.9311 | Val Loss: 0.2807, Acc: 0.9128
Epoch 03 | Train Loss: 0.1618, Acc: 0.9699 | Val Loss: 0.3168, Acc: 0.9262
Epoch 04 | Train Loss: 0.0981, Acc: 0.9813 | Val Loss: 0.2569, Acc: 0.9195
Epoch 05 | Train Loss: 0.0572, Acc: 0.9914 | Val Loss: 0.2671, Acc: 0.9262
Epoch 06 | Train Loss: 0.0453, Acc: 0.9900 | Val Loss: 0.3235, Acc: 0.9195
Epoch 07 | Train Loss: 0.0322, Acc: 0.9957 | Val Loss: 0.2617, Acc: 0.9396
Epoch 08 | Train Loss: 0.0211, Acc: 1.0000 | Val Loss: 0.2947, Acc: 0.9329
Epoch 09 | Train Loss: 0.0197, Acc: 0.9986 | Val Loss: 0.3198, Acc: 0.9396
Early stopping terpicu di epoch 9!


In [7]:
# 7. Evaluasi pada Test Set (Macro F1-Score)[cite: 4]
model.load_state_dict(best_model_wts)
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        all_preds.extend(torch.max(model(inputs), 1)[1].cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
test_f1 = f1_score(all_labels, all_preds, average='macro')
print(f"\n[HASIL AKHIR] Test Akurasi: {test_acc:.4f} | Test Macro F1-Score: {test_f1:.4f}")


[HASIL AKHIR] Test Akurasi: 0.9200 | Test Macro F1-Score: 0.9190


In [10]:
import copy
import time
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import f1_score
import numpy as np

def jalankan_eksperimen(kondisi, train_loader, val_loader, test_loader):
    print(f"\n{'='*40}")
    print(f"🚀 MEMULAI EKSPERIMEN: {kondisi}")
    print(f"{'='*40}")

    # 1. Atur Seed Ulang agar tiap eksperimen adil
    set_seed(67)

    # 2. INISIALISASI MODEL & OPTIMIZER SESUAI KONDISI
    if kondisi == 'B1':
        # Baseline from Scratch: Tanpa pretrained weights
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 10)
        model = model.to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

    elif kondisi == 'B2':
        # Feature Extraction: Bekukan semua kecuali FC
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        model = models.resnet18(weights=weights)
        for param in model.parameters(): param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, 10)
        model = model.to(device)
        optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-2)

    elif kondisi in ['B3', 'B4']:
        # Progressive Fine-Tuning: Buka layer4 dan FC
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        model = models.resnet18(weights=weights)
        for param in model.parameters(): param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, 10)
        for param in model.layer4.parameters(): param.requires_grad = True
        model = model.to(device)
        optimizer = torch.optim.AdamW([
            {"params": model.layer4.parameters(), "lr": 1e-4},
            {"params": model.fc.parameters(), "lr": 1e-3},
        ], weight_decay=1e-2)

    # Cek jumlah parameter
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Trainable Parameters: {trainable_params:,}")

    # 3. TRAINING LOOP
    criterion = nn.CrossEntropyLoss()
    num_epochs = 50
    patience = 5
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_wts = copy.deepcopy(model.state_dict())

    start_time = time.time()

    for epoch in range(num_epochs):
        model.train()
        train_loss, train_correct = 0.0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)
            train_correct += torch.sum(torch.max(outputs, 1)[1] == labels.data)

        model.eval()
        val_loss, val_correct = 0.0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_correct += torch.sum(torch.max(outputs, 1)[1] == labels.data)

        ep_t_loss = train_loss / len(train_loader.dataset)
        ep_t_acc = train_correct.double() / len(train_loader.dataset)
        ep_v_loss = val_loss / len(val_loader.dataset)
        ep_v_acc = val_correct.double() / len(val_loader.dataset)

        print(f"Epoch {epoch+1:02d} | Train Loss: {ep_t_loss:.4f}, Acc: {ep_t_acc:.4f} | Val Loss: {ep_v_loss:.4f}, Acc: {ep_v_acc:.4f}")

        # Early Stopping
        if ep_v_loss < best_val_loss:
            best_val_loss, best_model_wts, epochs_no_improve = ep_v_loss, copy.deepcopy(model.state_dict()), 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping terpicu di epoch {epoch+1}!")
                break

    total_time = time.time() - start_time
    epoch_berhenti = epoch + 1

    # 4. KEMBALIKAN METRIK UNTUK DICATAT
    model.load_state_dict(best_model_wts)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            all_preds.extend(torch.max(model(inputs), 1)[1].cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_acc = sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
    test_f1 = f1_score(all_labels, all_preds, average='macro')

    print(f"\n[HASIL AKHIR] Test Akurasi: {test_acc:.4f} | Test Macro F1-Score: {test_f1:.4f}")
    print(f"✅ Eksperimen {kondisi} Selesai!")
    return test_acc, test_f1, total_time, epoch_berhenti

In [11]:
# 1. Buat train_loader_tanpa_augmentasi khusus untuk B4 menggunakan eval_transforms
train_loader_tanpa_augmentasi = DataLoader(
    PetSubset(full_dataset, train_idx, eval_transforms),
    batch_size=32,
    shuffle=True
)

# Jalankan B1
b1_acc, b1_f1, b1_time, b1_epoch = jalankan_eksperimen('B1', train_loader, val_loader, test_loader)

# Jalankan B2
b2_acc, b2_f1, b2_time, b2_epoch = jalankan_eksperimen('B2', train_loader, val_loader, test_loader)



🚀 MEMULAI EKSPERIMEN: B1
Total Trainable Parameters: 11,181,642
Epoch 01 | Train Loss: 2.3873, Acc: 0.1621 | Val Loss: 9.8209, Acc: 0.1544
Epoch 02 | Train Loss: 2.1055, Acc: 0.2496 | Val Loss: 2.3163, Acc: 0.1745
Epoch 03 | Train Loss: 2.0250, Acc: 0.2496 | Val Loss: 2.1687, Acc: 0.2148
Epoch 04 | Train Loss: 1.9350, Acc: 0.3027 | Val Loss: 2.2739, Acc: 0.2215
Epoch 05 | Train Loss: 1.8821, Acc: 0.3286 | Val Loss: 2.7170, Acc: 0.2081
Epoch 06 | Train Loss: 1.8779, Acc: 0.3472 | Val Loss: 1.9136, Acc: 0.3423
Epoch 07 | Train Loss: 1.8120, Acc: 0.3702 | Val Loss: 1.9560, Acc: 0.3020
Epoch 08 | Train Loss: 1.6875, Acc: 0.3960 | Val Loss: 1.9615, Acc: 0.3020
Epoch 09 | Train Loss: 1.7674, Acc: 0.3702 | Val Loss: 1.9654, Acc: 0.3624
Epoch 10 | Train Loss: 1.6718, Acc: 0.4204 | Val Loss: 1.9248, Acc: 0.3289
Epoch 11 | Train Loss: 1.7058, Acc: 0.4075 | Val Loss: 1.8110, Acc: 0.3624
Epoch 12 | Train Loss: 1.7037, Acc: 0.4118 | Val Loss: 1.9183, Acc: 0.3221
Epoch 13 | Train Loss: 1.5321, Acc:

In [12]:
# Jalankan B3
b3_acc, b3_f1, b3_time, b3_epoch = jalankan_eksperimen('B3', train_loader, val_loader, test_loader)

# Jalankan B4 (Uji Kontrol tanpa augmentasi)
# Gunakan dataloader khusus B4
b4_acc, b4_f1, b4_time, b4_epoch = jalankan_eksperimen('B4', train_loader_tanpa_augmentasi, val_loader, test_loader)


🚀 MEMULAI EKSPERIMEN: B3
Total Trainable Parameters: 8,398,858
Epoch 01 | Train Loss: 1.2678, Acc: 0.6356 | Val Loss: 0.3915, Acc: 0.8725
Epoch 02 | Train Loss: 0.3090, Acc: 0.9311 | Val Loss: 0.2807, Acc: 0.9128
Epoch 03 | Train Loss: 0.1618, Acc: 0.9699 | Val Loss: 0.3168, Acc: 0.9262
Epoch 04 | Train Loss: 0.0981, Acc: 0.9813 | Val Loss: 0.2569, Acc: 0.9195
Epoch 05 | Train Loss: 0.0572, Acc: 0.9914 | Val Loss: 0.2671, Acc: 0.9262
Epoch 06 | Train Loss: 0.0453, Acc: 0.9900 | Val Loss: 0.3235, Acc: 0.9195
Epoch 07 | Train Loss: 0.0322, Acc: 0.9957 | Val Loss: 0.2617, Acc: 0.9396
Epoch 08 | Train Loss: 0.0211, Acc: 1.0000 | Val Loss: 0.2947, Acc: 0.9329
Epoch 09 | Train Loss: 0.0197, Acc: 0.9986 | Val Loss: 0.3198, Acc: 0.9396
Early stopping terpicu di epoch 9!

[HASIL AKHIR] Test Akurasi: 0.9200 | Test Macro F1-Score: 0.9190
✅ Eksperimen B3 Selesai!

🚀 MEMULAI EKSPERIMEN: B4
Total Trainable Parameters: 8,398,858
Epoch 01 | Train Loss: 1.1404, Acc: 0.6829 | Val Loss: 0.3838, Acc: 0.8

In [13]:
import pandas as pd
import re

# Ini teks log yang kamu copas tadi
raw_log = """
========================================
🚀 MEMULAI EKSPERIMEN: B1
========================================
Epoch 01 | Train Loss: 2.3873, Acc: 0.1621 | Val Loss: 9.8209, Acc: 0.1544
Epoch 02 | Train Loss: 2.1055, Acc: 0.2496 | Val Loss: 2.3163, Acc: 0.1745
Epoch 03 | Train Loss: 2.0250, Acc: 0.2496 | Val Loss: 2.1687, Acc: 0.2148
Epoch 04 | Train Loss: 1.9350, Acc: 0.3027 | Val Loss: 2.2739, Acc: 0.2215
Epoch 05 | Train Loss: 1.8821, Acc: 0.3286 | Val Loss: 2.7170, Acc: 0.2081
Epoch 06 | Train Loss: 1.8779, Acc: 0.3472 | Val Loss: 1.9136, Acc: 0.3423
Epoch 07 | Train Loss: 1.8120, Acc: 0.3702 | Val Loss: 1.9560, Acc: 0.3020
Epoch 08 | Train Loss: 1.6875, Acc: 0.3960 | Val Loss: 1.9615, Acc: 0.3020
Epoch 09 | Train Loss: 1.7674, Acc: 0.3702 | Val Loss: 1.9654, Acc: 0.3624
Epoch 10 | Train Loss: 1.6718, Acc: 0.4204 | Val Loss: 1.9248, Acc: 0.3289
Epoch 11 | Train Loss: 1.7058, Acc: 0.4075 | Val Loss: 1.8110, Acc: 0.3624
Epoch 12 | Train Loss: 1.7037, Acc: 0.4118 | Val Loss: 1.9183, Acc: 0.3221
Epoch 13 | Train Loss: 1.5321, Acc: 0.4806 | Val Loss: 2.4030, Acc: 0.2886
Epoch 14 | Train Loss: 1.5790, Acc: 0.4491 | Val Loss: 2.2490, Acc: 0.3490
Epoch 15 | Train Loss: 1.6024, Acc: 0.4505 | Val Loss: 2.0701, Acc: 0.3221
Epoch 16 | Train Loss: 1.4096, Acc: 0.5308 | Val Loss: 2.0110, Acc: 0.3826

========================================
🚀 MEMULAI EKSPERIMEN: B2
========================================
Epoch 01 | Train Loss: 2.0198, Acc: 0.2884 | Val Loss: 1.3666, Acc: 0.6107
Epoch 02 | Train Loss: 1.1606, Acc: 0.7547 | Val Loss: 0.8316, Acc: 0.8993
Epoch 03 | Train Loss: 0.8018, Acc: 0.8422 | Val Loss: 0.5970, Acc: 0.8792
Epoch 04 | Train Loss: 0.6128, Acc: 0.8867 | Val Loss: 0.5091, Acc: 0.8993
Epoch 05 | Train Loss: 0.4995, Acc: 0.9082 | Val Loss: 0.4404, Acc: 0.8993
Epoch 06 | Train Loss: 0.4571, Acc: 0.9053 | Val Loss: 0.3904, Acc: 0.8859
Epoch 07 | Train Loss: 0.3702, Acc: 0.9340 | Val Loss: 0.3560, Acc: 0.8993
Epoch 08 | Train Loss: 0.3544, Acc: 0.9326 | Val Loss: 0.3391, Acc: 0.8859
Epoch 09 | Train Loss: 0.3366, Acc: 0.9326 | Val Loss: 0.3314, Acc: 0.8993
Epoch 10 | Train Loss: 0.3226, Acc: 0.9297 | Val Loss: 0.3045, Acc: 0.9128
Epoch 11 | Train Loss: 0.2715, Acc: 0.9397 | Val Loss: 0.2949, Acc: 0.9128
Epoch 12 | Train Loss: 0.2757, Acc: 0.9469 | Val Loss: 0.2847, Acc: 0.8926
Epoch 13 | Train Loss: 0.2633, Acc: 0.9426 | Val Loss: 0.2719, Acc: 0.9262
Epoch 14 | Train Loss: 0.2447, Acc: 0.9512 | Val Loss: 0.2792, Acc: 0.9262
Epoch 15 | Train Loss: 0.2184, Acc: 0.9555 | Val Loss: 0.2779, Acc: 0.9195
Epoch 16 | Train Loss: 0.2053, Acc: 0.9498 | Val Loss: 0.2721, Acc: 0.9060
Epoch 17 | Train Loss: 0.2203, Acc: 0.9541 | Val Loss: 0.2962, Acc: 0.9195
Epoch 18 | Train Loss: 0.2227, Acc: 0.9498 | Val Loss: 0.2522, Acc: 0.9128
Epoch 19 | Train Loss: 0.1919, Acc: 0.9555 | Val Loss: 0.2624, Acc: 0.9128
Epoch 20 | Train Loss: 0.1871, Acc: 0.9627 | Val Loss: 0.2713, Acc: 0.9128
Epoch 21 | Train Loss: 0.1913, Acc: 0.9584 | Val Loss: 0.2661, Acc: 0.9128
Epoch 22 | Train Loss: 0.1922, Acc: 0.9555 | Val Loss: 0.2584, Acc: 0.9195
Epoch 23 | Train Loss: 0.1675, Acc: 0.9613 | Val Loss: 0.2752, Acc: 0.9195

========================================
🚀 MEMULAI EKSPERIMEN: B3
========================================
Epoch 01 | Train Loss: 1.2678, Acc: 0.6356 | Val Loss: 0.3915, Acc: 0.8725
Epoch 02 | Train Loss: 0.3090, Acc: 0.9311 | Val Loss: 0.2807, Acc: 0.9128
Epoch 03 | Train Loss: 0.1618, Acc: 0.9699 | Val Loss: 0.3168, Acc: 0.9262
Epoch 04 | Train Loss: 0.0981, Acc: 0.9813 | Val Loss: 0.2569, Acc: 0.9195
Epoch 05 | Train Loss: 0.0572, Acc: 0.9914 | Val Loss: 0.2671, Acc: 0.9262
Epoch 06 | Train Loss: 0.0453, Acc: 0.9900 | Val Loss: 0.3235, Acc: 0.9195
Epoch 07 | Train Loss: 0.0322, Acc: 0.9957 | Val Loss: 0.2617, Acc: 0.9396
Epoch 08 | Train Loss: 0.0211, Acc: 1.0000 | Val Loss: 0.2947, Acc: 0.9329
Epoch 09 | Train Loss: 0.0197, Acc: 0.9986 | Val Loss: 0.3198, Acc: 0.9396

========================================
🚀 MEMULAI EKSPERIMEN: B4
========================================
Epoch 01 | Train Loss: 1.1404, Acc: 0.6829 | Val Loss: 0.3838, Acc: 0.8658
Epoch 02 | Train Loss: 0.1826, Acc: 0.9727 | Val Loss: 0.2972, Acc: 0.9128
Epoch 03 | Train Loss: 0.0628, Acc: 0.9928 | Val Loss: 0.2610, Acc: 0.9262
Epoch 04 | Train Loss: 0.0292, Acc: 1.0000 | Val Loss: 0.2744, Acc: 0.9195
Epoch 05 | Train Loss: 0.0166, Acc: 1.0000 | Val Loss: 0.2697, Acc: 0.9262
Epoch 06 | Train Loss: 0.0115, Acc: 1.0000 | Val Loss: 0.2657, Acc: 0.9262
Epoch 07 | Train Loss: 0.0085, Acc: 1.0000 | Val Loss: 0.2678, Acc: 0.9195
Epoch 08 | Train Loss: 0.0060, Acc: 1.0000 | Val Loss: 0.2675, Acc: 0.9262
"""

data = []
current_exp = ""

# Ekstrak data pakai Regex
for line in raw_log.strip().split('\n'):
    if "MEMULAI EKSPERIMEN:" in line:
        current_exp = line.split(":")[-1].strip()
    elif line.startswith("Epoch"):
        match = re.search(r"Epoch (\d+) \| Train Loss: ([\d.]+), Acc: ([\d.]+) \| Val Loss: ([\d.]+), Acc: ([\d.]+)", line)
        if match:
            data.append({
                "Eksperimen": current_exp,
                "Epoch": int(match.group(1)),
                "Train Loss": float(match.group(2)),
                "Train Acc": float(match.group(3)),
                "Val Loss": float(match.group(4)),
                "Val Acc": float(match.group(5))
            })

# Jadikan DataFrame dan simpan ke CSV
df = pd.DataFrame(data)
df.to_csv("log_hasil_eksperimen.csv", index=False)
print("✅ CSV Berhasil dibuat! Cek tab folder di kiri Colab kamu.")
df.head()

✅ CSV Berhasil dibuat! Cek tab folder di kiri Colab kamu.


,Eksperimen,Epoch,Train Loss,Train Acc,Val Loss,Val Acc
0,B1,1,2.3873,0.1621,9.8209,0.1544
1,B1,2,2.1055,0.2496,2.3163,0.1745
2,B1,3,2.0250,0.2496,2.1687,0.2148
3,B1,4,1.9350,0.3027,2.2739,0.2215
4,B1,5,1.8821,0.3286,2.7170,0.2081
